# NumPy — Complete Revision Notebook

A clean, topic-wise reference of NumPy for data analysis and Machine Learning / AI.
Each section has: **what it is → why it matters → runnable example**.
Notes from the original practice notebook are folded into the explanations.

## 1. What is NumPy and why it matters for ML/AI

NumPy (`Numerical Python`) is the core library for fast numerical computation in Python.
Almost every ML/DL library (pandas, scikit-learn, PyTorch, TensorFlow) is either built on
NumPy arrays or uses the exact same mental model (n-dimensional arrays, vectorized math,
broadcasting). Learning NumPy well = learning the "assembly language" of ML.

Key reasons it's crucial:
- **Speed**: vectorized operations run in optimized C loops instead of slow Python `for` loops.
- **Memory efficiency**: arrays store data in contiguous, typed memory blocks.
- **Foundation**: tensors in PyTorch/TensorFlow behave almost identically to NumPy arrays.

In [1]:
import numpy as np
print(np.__version__)

2.4.4


## 2. Creating Arrays

| Function | Purpose |
|---|---|
| `np.array()` | Convert a Python list/tuple into an ndarray |
| `np.zeros()` | Array filled with 0s (init weights, placeholders, masks) |
| `np.ones()` | Array filled with 1s (bias terms, default values) |
| `np.full()` | Array filled with a constant |
| `np.eye()` | Identity matrix (diagonal of 1s) |
| `np.empty()` | Uninitialized array (fast, garbage values — use only when you will overwrite every value) |
| `np.arange()` | Like Python's `range()` but returns an actual array, good for numeric math |
| `np.linspace()` | Evenly spaced numbers between start and stop (inclusive) — great for plotting, bins |

In [2]:
a = np.array(42)                       # 0-D (scalar)
b = np.array([1, 2, 3, 4, 8])          # 1-D
c = np.array([[1, 2, 3], [4, 5, 6]])   # 2-D
d = np.array([[[1, 2, 3], [4, 5, 6]], [[1, 2, 3], [4, 5, 6]]])  # 3-D

print("ndim a,b,c,d:", a.ndim, b.ndim, c.ndim, d.ndim)
print("b[4] ->", b[4])
print("shape of d:", d.shape, "| size:", d.size, "| dtype:", d.dtype)

ndim a,b,c,d: 0 1 2 3
b[4] -> 8
shape of d: (2, 2, 3) | size: 12 | dtype: int64


In [3]:
Z = np.zeros((3, 4))          # placeholder / init weights
O = np.ones((2, 3))           # bias / default values
F = np.full((3, 3), 7)        # constant-filled array
E = np.eye(3)                 # identity matrix -> used in linear algebra, one-hot tricks
J = np.empty(3)                # uninitialized (fast) memory

print("zeros:\n", Z)
print("ones:\n", O)
print("full:\n", F)
print("eye:\n", E)
print("empty (garbage values):", J)

zeros:
 [[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
ones:
 [[1. 1. 1.]
 [1. 1. 1.]]
full:
 [[7 7 7]
 [7 7 7]
 [7 7 7]]
eye:
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
empty (garbage values): [6.90786283e-310 1.17312014e-315 3.57725758e-057]


In [4]:
# range() vs np.arange()
r = range(10)
a = np.arange(12)
print(type(r), type(a))
# range()   -> plain Python looping object
# np.arange() -> real ndarray, supports vectorized math (a * 2, a + 1, etc.)
print(a * 2)

<class 'range'> <class 'numpy.ndarray'>
[ 0  2  4  6  8 10 12 14 16 18 20 22]


In [5]:
# np.linspace(start, stop, num) -> evenly spaced values, INCLUSIVE of stop
print(np.linspace(0, 1, 5))
print(np.linspace(-1, 1, 5))
print(np.linspace(0, 100, 11))          # e.g. building histogram bin edges
print(np.linspace(0, 1, 5, dtype=int))  # careful: casting can collapse values to 0/1

[0.   0.25 0.5  0.75 1.  ]
[-1.  -0.5  0.   0.5  1. ]
[  0.  10.  20.  30.  40.  50.  60.  70.  80.  90. 100.]
[0 0 0 0 1]


## 3. Data Types (`dtype`) and Type Conversion

NumPy arrays are **homogeneous** — every element shares one dtype. If you mix types
(e.g. numbers + a string), NumPy upcasts everything to the most general type (usually
a Unicode string `<U..`). This matters in ML because feeding the wrong dtype into a
model (e.g. strings instead of floats) causes errors or silently wrong math.

In [6]:
a = np.array([1.1, 2, 3, 4, 5])
print(a.dtype)                     # float64

b = np.array([1, 2, 3, 4.4, 'siffat'])
print(b.dtype, b)                  # everything upcast to '<U32' (unicode string)
# U = Unicode string, the number = max characters reserved per element

float64
<U32 ['1' '2' '3' '4.4' 'siffat']


In [7]:
# Force number of dimensions
a = np.array([2, 3, 4, 5, 6], ndmin=5)
print(a)
print(a.ndim)

[[[[[2 3 4 5 6]]]]]
5


In [8]:
# Explicit dtype at creation
arr = np.array([1, 2, 3, 4], dtype='S')   # byte-string
print(arr, arr.dtype)

arr1 = np.array([1, 2, 3, 4], dtype='i4')  # 4-byte (32-bit) integer
print(arr1, arr1.dtype)
# For i (int), u (unsigned int), f (float), S, U you can specify a size, e.g. 'i8', 'f4'

[b'1' b'2' b'3' b'4'] |S1
[1 2 3 4] int32


In [9]:
# astype() -> returns a NEW array with a different dtype (does not modify original)
arr = np.array([1.1, 2.1, 3.1])
newarr = arr.astype(int)          # or arr.astype('i')
print(newarr, newarr.dtype)

# Very common ML use: converting labels to int, or pixel values to float32
labels = np.array(['0', '1', '1', '0'])
print(labels.astype(int))

[1 2 3] int64
[0 1 1 0]


## 4. Indexing & Slicing

- **Indexing** returns a single value.
- **Slicing** returns a *view* (range) of values: `arr[start:end]` — `end` is exclusive.

For 2D/3D arrays, use comma-separated indices: `arr[row, col]`, `arr[dim0, dim1, dim2]`.

| Syntax | Meaning |
|---|---|
| `arr[i, j]` | single element at row i, column j |
| `arr[i, :]` | entire row i |
| `arr[:, j]` | entire column j |
| `arr[a:b, c:d]` | sub-matrix, end exclusive |

In [10]:
arr = np.array([[1, 2, 3, 4, 5], [6, 7, 8, 9, 10]])
print("2nd element, 1st row:", arr[0, 1])
print("last element, 2nd row:", arr[1, -1])
print("row slice:", arr[1, 1:4])       # from index 1 up to (not incl.) 4
print("full column 2:", arr[:, 2])

2nd element, 1st row: 2
last element, 2nd row: 10
row slice: [7 8 9]
full column 2: [3 8]


In [11]:
arr3d = np.array([[[1, 2, 3], [4, 5, 6]], [[7, 8, 9], [10, 11, 12]]])
print(arr3d[0, 1, 2])   # first block, 2nd row, 3rd column

6


In [12]:
# Boolean / fancy indexing -> the backbone of filtering in ML preprocessing
arr = np.array([10, 15, 20, 25, 30])
mask = arr > 18
print(mask)
print(arr[mask])              # only elements where mask is True
print(arr[[0, 2, 4]])         # fancy indexing: pick specific positions

[False False  True  True  True]
[20 25 30]
[10 20 30]


## 5. Copy vs View

- `.copy()` -> independent new array; changing the original does **not** affect the copy.
- `.view()` -> shares the same underlying memory; changing the original **does** affect the view.

This matters a lot in ML pipelines: accidentally mutating a "view" of your dataset can
silently corrupt your original data.

In [13]:
arr = np.array([1, 2, 3, 4, 5])
x = arr.copy()
arr[0] = 42
print("original:", arr, "| copy (unaffected):", x)

arr = np.array([1, 2, 3, 4, 5])
y = arr.view()
arr[0] = 42
print("original:", arr, "| view (also changed):", y)

original: [42  2  3  4  5] | copy (unaffected): [1 2 3 4 5]
original: [42  2  3  4  5] | view (also changed): [42  2  3  4  5]


## 6. Mathematical & Statistical Functions

These are used constantly for feature engineering, normalization, and evaluating models.

In [14]:
arr = np.array([1, 2, 3, 4])

print("sqrt:", np.sqrt(arr))
print("exp :", np.exp(arr))
print("log :", np.log(arr))       # natural log
print("sin :", np.sin(arr))
print("sum :", np.sum(arr))
print("mean:", np.mean(arr))
print("max :", np.max(arr))
print("min :", np.min(arr))
print("std :", np.std(arr))
print("var :", np.var(arr))
print("median:", np.median(arr))
print("cumsum :", np.cumsum(arr))   # running total -> useful for cumulative metrics
print("cumprod:", np.cumprod(arr))

sqrt: [1.         1.41421356 1.73205081 2.        ]
exp : [ 2.71828183  7.3890561  20.08553692 54.59815003]
log : [0.         0.69314718 1.09861229 1.38629436]
sin : [ 0.84147098  0.90929743  0.14112001 -0.7568025 ]
sum : 10
mean: 2.5
max : 4
min : 1
std : 1.118033988749895
var : 1.25
median: 2.5
cumsum : [ 1  3  6 10]
cumprod: [ 1  2  6 24]


In [15]:
arr = np.array([10, 20, 30, 40, 50, 60, 70, 80])
print("25th percentile:", np.percentile(arr, 25))
print("50th percentile (median):", np.percentile(arr, 50))
print("75th percentile:", np.percentile(arr, 75))
# Percentiles are used to detect outliers (IQR method) before training a model

25th percentile: 27.5
50th percentile (median): 45.0
75th percentile: 62.5


## 7. Reshaping, Stacking, and Splitting

Reshaping is essential in ML: e.g. flattening an image matrix into a feature vector,
or reshaping a 1D prediction array into a column vector before feeding a model.

In [16]:
arr = np.arange(12)
print("original:", arr)
reshaped = arr.reshape(3, 4)
print("reshaped 3x4:\n", reshaped)

print("flatten back to 1D:", reshaped.flatten())
print("transpose:\n", reshaped.T)     # rows <-> columns, used constantly in linear algebra

original: [ 0  1  2  3  4  5  6  7  8  9 10 11]
reshaped 3x4:
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
flatten back to 1D: [ 0  1  2  3  4  5  6  7  8  9 10 11]
transpose:
 [[ 0  4  8]
 [ 1  5  9]
 [ 2  6 10]
 [ 3  7 11]]


In [17]:
a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6], [7, 8]])

print("vstack (stack rows):\n", np.vstack((a, b)))
print("hstack (stack columns):\n", np.hstack((a, b)))
print("concatenate axis=0:\n", np.concatenate((a, b), axis=0))

vstack (stack rows):
 [[1 2]
 [3 4]
 [5 6]
 [7 8]]
hstack (stack columns):
 [[1 2 5 6]
 [3 4 7 8]]
concatenate axis=0:
 [[1 2]
 [3 4]
 [5 6]
 [7 8]]


In [18]:
arr = np.array([[1, 2], [3, 4], [5, 6], [7, 8]])

split_rows = np.array_split(arr, 2, axis=0)
print("split along rows:")
for part in split_rows:
    print(part)

split_cols = np.array_split(arr, 2, axis=1)
print("split along columns:")
for part in split_cols:
    print(part)

split along rows:
[[1 2]
 [3 4]]
[[5 6]
 [7 8]]
split along columns:
[[1]
 [3]
 [5]
 [7]]
[[2]
 [4]
 [6]
 [8]]


## 8. Broadcasting

Broadcasting lets NumPy perform element-wise operations on arrays of **different shapes**
without writing explicit loops, by "stretching" the smaller array. This is exactly how
frameworks like PyTorch add a bias vector to every row of a batch of data.

**Rule**: dimensions are compatible if they're equal, or one of them is 1. If neither holds,
broadcasting fails.

In [19]:
# 1) scalar with array
a = np.array([1, 2, 3])
print(a + 5)

# 2) 1D array with 2D array (broadcast across rows)
A = np.array([[1, 2, 3], [4, 5, 6]])
b = np.array([10, 20, 30])
print(A + b)

# 3) column vector broadcasting (broadcast across columns)
c = np.array([[10], [20]])
print(A + c)

# 4) when broadcasting FAILS (shapes incompatible)
try:
    d = np.array([1, 2])
    print(A + d)
except ValueError as e:
    print("Broadcasting error:", e)

[6 7 8]
[[11 22 33]
 [14 25 36]]
[[11 12 13]
 [24 25 26]]
Broadcasting error: operands could not be broadcast together with shapes (2,3) (2,) 


## 9. The `np.random` Module

Used everywhere in ML: initializing weights, shuffling datasets, train/test splitting,
generating synthetic data, dropout masks, etc.

| Function | Purpose |
|---|---|
| `np.random.seed()` | Fix randomness for reproducible results |
| `np.random.rand()` | Uniform [0,1), shape as separate args |
| `np.random.random()` | Uniform [0,1), shape as a tuple |
| `np.random.randn()` | Standard normal (Gaussian) distribution |
| `np.random.randint()` | Random integers in a range |
| `np.random.choice()` | Random sample from a given array |
| `np.random.shuffle()` | Shuffle an array in place |

In [20]:
np.random.seed(42)   # reproducibility -> same "random" numbers every run

random_values = np.random.random((3, 3))   # shape passed as a TUPLE
print("random():\n", random_values)

random_rand = np.random.rand(3, 3)         # shape passed as separate args
print("rand():\n", random_rand)

print("randn (Gaussian):\n", np.random.randn(3))
print("randint(0,10,size=5):", np.random.randint(0, 10, size=5))
print("choice:", np.random.choice([10, 20, 30, 40], size=3))

arr = np.arange(10)
np.random.shuffle(arr)      # shuffles IN PLACE -> common before train/test split
print("shuffled:", arr)

random():
 [[0.37454012 0.95071431 0.73199394]
 [0.59865848 0.15601864 0.15599452]
 [0.05808361 0.86617615 0.60111501]]
rand():
 [[0.70807258 0.02058449 0.96990985]
 [0.83244264 0.21233911 0.18182497]
 [0.18340451 0.30424224 0.52475643]]
randn (Gaussian):
 [-1.72491783 -0.56228753 -1.01283112]
randint(0,10,size=5): [2 6 3 8 2]
choice: [10 30 30]
shuffled: [9 2 7 5 0 3 1 6 8 4]


## 10. Linear Algebra (`np.linalg`) — *added, missing from the original notes*

Linear algebra is the mathematical backbone of ML: dot products power neural network
layers, matrix inverses appear in linear regression's closed-form solution, and
eigenvectors drive PCA (dimensionality reduction).

In [21]:
A = np.array([[2, 0], [0, 3]])
B = np.array([[1, 2], [3, 4]])

print("dot product:\n", np.dot(A, B))       # matrix multiplication
print("matmul (same for 2D):\n", A @ B)      # @ is shorthand for matmul
print("determinant:", np.linalg.det(B))
print("inverse:\n", np.linalg.inv(B))
eigvals, eigvecs = np.linalg.eig(B)
print("eigenvalues:", eigvals)
print("eigenvectors:\n", eigvecs)
print("norm (vector length):", np.linalg.norm(np.array([3, 4])))   # used in distance metrics, regularization

dot product:
 [[ 2  4]
 [ 9 12]]
matmul (same for 2D):
 [[ 2  4]
 [ 9 12]]
determinant: -2.0000000000000004
inverse:
 [[-2.   1. ]
 [ 1.5 -0.5]]
eigenvalues: [-0.37228132  5.37228132]
eigenvectors:
 [[-0.82456484 -0.41597356]
 [ 0.56576746 -0.90937671]]
norm (vector length): 5.0


## 11. Sorting, Uniqueness, and Argmax/Argmin — *added*

`argmax`/`argmin` are used constantly in ML to convert model probability outputs into
predicted class labels (e.g. `np.argmax(softmax_output)`).

In [22]:
arr = np.array([3, 1, 4, 1, 5, 9, 2, 6])

print("sorted:", np.sort(arr))
print("argsort (indices that would sort):", np.argsort(arr))
print("unique values:", np.unique(arr))
print("unique + counts:", np.unique(arr, return_counts=True))

print("index of max:", np.argmax(arr))   # e.g. picking the predicted class
print("index of min:", np.argmin(arr))

probs = np.array([0.1, 0.7, 0.2])
print("predicted class:", np.argmax(probs))

sorted: [1 1 2 3 4 5 6 9]
argsort (indices that would sort): [1 3 6 0 2 4 7 5]
unique values: [1 2 3 4 5 6 9]
unique + counts: (array([1, 2, 3, 4, 5, 6, 9]), array([2, 1, 1, 1, 1, 1, 1]))
index of max: 5
index of min: 1
predicted class: 1


## 12. Conditional Selection, Clipping, and Missing Values — *added*

`np.where` is a vectorized if/else — extremely common for labeling/thresholding.
`np.clip` keeps values within bounds (used to prevent exploding gradients / cap outliers).
`np.isnan` + friends are essential for cleaning real-world (messy) datasets before training.

In [23]:
arr = np.array([1, -2, 3, -4, 5])
print("where(arr>0, arr, 0):", np.where(arr > 0, arr, 0))      # ReLU-like operation!

print("clip between 0 and 3:", np.clip(arr, 0, 3))

data = np.array([1.0, np.nan, 3.0, np.nan, 5.0])
print("isnan mask:", np.isnan(data))
print("count of NaNs:", np.isnan(data).sum())
print("nanmean (ignores NaN):", np.nanmean(data))
clean = data[~np.isnan(data)]
print("data with NaNs removed:", clean)

where(arr>0, arr, 0): [1 0 3 0 5]
clip between 0 and 3: [1 0 3 0 3]
isnan mask: [False  True False  True False]
count of NaNs: 2
nanmean (ignores NaN): 3.0
data with NaNs removed: [1. 3. 5.]


## 13. Why Vectorization Matters — *added*

This is the single biggest practical reason NumPy exists for ML: replacing Python
`for` loops with vectorized array operations can be 50-100x+ faster, which matters
enormously when processing millions of training examples.

In [24]:
import time

n = 1_000_000
x = list(range(n))
y = np.arange(n)

start = time.time()
result_loop = [i * 2 for i in x]
loop_time = time.time() - start

start = time.time()
result_vec = y * 2
vec_time = time.time() - start

print(f"Python loop time : {loop_time:.5f}s")
print(f"NumPy vectorized : {vec_time:.5f}s")
print(f"Speedup: ~{loop_time/vec_time:.1f}x faster")

Python loop time : 0.32087s
NumPy vectorized : 0.05047s
Speedup: ~6.4x faster


## 14. Extra Utilities Crucial for ML/AI — *added*

- `np.newaxis` — adds a dimension (e.g. turning a 1D feature vector into a 2D "column" a model expects).
- `np.eye()` for **one-hot encoding** class labels.
- Feature **standardization** (mean 0, std 1) — the most common preprocessing step before training.
- Saving/loading arrays to disk (`np.save` / `np.load`) — used to cache preprocessed datasets or model weights.

In [25]:
x = np.array([1, 2, 3])
print("original shape:", x.shape)
print("with newaxis (row):", x[np.newaxis, :].shape)
print("with newaxis (col):", x[:, np.newaxis].shape)

original shape: (3,)
with newaxis (row): (1, 3)
with newaxis (col): (3, 1)


In [26]:
# One-hot encoding using np.eye -> common label-encoding trick for classification
num_classes = 4
labels = np.array([0, 2, 1, 3])
one_hot = np.eye(num_classes)[labels]
print(one_hot)

[[1. 0. 0. 0.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]]


In [27]:
# Standardization (z-score normalization): critical preprocessing step for most ML models
data = np.array([10, 20, 30, 40, 50], dtype=float)
standardized = (data - data.mean()) / data.std()
print("standardized:", standardized)

# Min-max normalization -> scales data to [0, 1]
normalized = (data - data.min()) / (data.max() - data.min())
print("normalized:", normalized)

standardized: [-1.41421356 -0.70710678  0.          0.70710678  1.41421356]
normalized: [0.   0.25 0.5  0.75 1.  ]


In [28]:
# Saving and loading arrays (useful for caching preprocessed data / model weights)
arr = np.array([1, 2, 3, 4, 5])
np.save('/tmp/my_array.npy', arr)
loaded = np.load('/tmp/my_array.npy')
print("loaded array:", loaded)

loaded array: [1 2 3 4 5]


## Summary — Original Notes vs What Was Added

**Covered from your original notebook:** array creation, ndim/shape/size/dtype, dtype
handling & astype, copy vs view, indexing/slicing (1D/2D/3D), reshape, vstack/hstack,
array_split, percentile, math/stat functions, broadcasting, random module basics.

**Added because they're essential for ML/AI and were missing:**
1. `np.linalg` — dot/matmul, inverse, determinant, eigenvalues (regression, PCA, neural nets)
2. `argmax` / `argmin` / `argsort` / `unique` — converting model outputs to predictions
3. `np.where`, `np.clip` — vectorized conditionals, outlier/gradient clipping
4. `np.isnan`, `np.nanmean` — handling missing data before training
5. Vectorization vs loops speed comparison — *why* NumPy matters at all
6. `np.newaxis`, one-hot encoding via `np.eye`, standardization/normalization — direct
   preprocessing steps used before feeding data into any ML model
7. `np.save` / `np.load` — persisting arrays/weights to disk